In [ ]:
import pandas as pd
import json

events_raw = pd.read_csv('data/events.csv')
print(len(events_raw.game_id.unique()))

# read cs_gold.json 
with open('data/cs_gold.json') as f:
    cs_gold = json.load(f)
print(len(cs_gold))

In [ ]:
events_raw[events_raw.game_id == 45323]

In [ ]:
# példa: requirements: pandas, numpy
import pandas as pd
import numpy as np

def parse_time_to_seconds(t):
    # t példa: "3:11" vagy "53:48"
    try:
        m,s = t.split(':')
        return int(m)*60 + int(s)
    except:
        return np.nan

def build_snapshots_for_game(game_id, events_raw, cs_gold, game_result_side):
    """
    game_id: int/str
    events_raw: DataFrame az egész datasetből (szűrni fogjuk a game_id-re)
    cs_gold: dict, cs_gold[str(game_id)] a te struktúrád szerint
    game_result_side: 'BLUE' vagy 'RED' -- ez az oldal amelyik nyert (vagy használj boolean home_win)
    """
    ev = events_raw[events_raw.game_id == game_id].copy()
    ev['secs'] = ev['time'].map(parse_time_to_seconds).astype('Int64')
    ev = ev.sort_values('secs')

    # maximális perc
    max_sec = int(ev.secs.max() if ev.secs.notna().any() else 0)
    max_min = max_sec // 60
    # ha cs_gold tartalmaz per-minute indexeket (labels), akkor használhatjuk azt a hosszot is
    csg = cs_gold.get(str(game_id), None)
    if csg:
        labels = csg['gold']['labels']
        max_min = max(max_min, len(labels)-1)

    minutes = list(range(0, max_min+1))
    rows = []
    # cumulative counters
    cum = {
        'kills_BLUE': 0, 'kills_RED': 0,
        'towers_BLUE': 0, 'towers_RED': 0,
        'drakes_BLUE': 0, 'drakes_RED': 0,
        'barons_BLUE': 0, 'barons_RED': 0,
        'plates_BLUE': 0, 'plates_RED': 0
    }
    last_obj_time = {'drake': None, 'baron': None}

    # előfeldolgozás: per event minute
    ev['minute'] = (ev['secs'] // 60).astype('Int64')
    events_by_minute = {m: ev[ev.minute == m] for m in minutes}

    # ha cs_gold adott: kinyerjük a lane cs/gold per minute
    # feltételezem cs_gold struktúra: cs_gold[str(game_id)]['gold']['datasets'] és ['cs']['datasets']
    lane_gold_by_min = {}
    lane_cs_by_min = {}
    if csg:
        gold_ds = csg['gold']['datasets']   # list of dicts label + data
        cs_ds = csg['cs']['datasets']
        # feltételezve, hogy minden ds.data hossza >= max_min+1
        for ds in gold_ds:
            role = ds['label']
            lane_gold_by_min[role] = ds['data']
        for ds in cs_ds:
            role = ds['label']
            lane_cs_by_min[role] = ds['data']

    for m in minutes:
        minute_events = events_by_minute.get(m, pd.DataFrame())
        # per-minute increments
        kills_blue_m = int(((minute_events.side == 'BLUE') & (minute_events.action == 'KILL')).sum())
        kills_red_m = int(((minute_events.side == 'RED') & (minute_events.action == 'KILL')).sum())
        cum['kills_BLUE'] += kills_blue_m
        cum['kills_RED'] += kills_red_m

        # torrevent / tower / TOWER in action
        towers_blue_m = int(((minute_events.side == 'BLUE') & (minute_events.action == 'TOWER')).sum())
        towers_red_m = int(((minute_events.side == 'RED') & (minute_events.action == 'TOWER')).sum())
        cum['towers_BLUE'] += towers_blue_m
        cum['towers_RED'] += towers_red_m

        # drake/baron
        drakes_blue_m = int(((minute_events.side == 'BLUE') & (minute_events.action == 'DRAKE')).sum())
        drakes_red_m = int(((minute_events.side == 'RED') & (minute_events.action == 'DRAKE')).sum())
        cum['drakes_BLUE'] += drakes_blue_m
        cum['drakes_RED'] += drakes_red_m
        # ha van DRAKE esemény, frissítjük last_obj_time
        if drakes_blue_m + drakes_red_m > 0:
            last_obj_time['drake'] = m

        barons_blue_m = int(((minute_events.side == 'BLUE') & (minute_events.action == 'BARON')).sum())
        barons_red_m = int(((minute_events.side == 'RED') & (minute_events.action == 'BARON')).sum())
        cum['barons_BLUE'] += barons_blue_m
        cum['barons_RED'] += barons_red_m
        if barons_blue_m + barons_red_m > 0:
            last_obj_time['baron'] = m

        plates_blue_m = int(((minute_events.side == 'BLUE') & (minute_events.action == 'PLATE')).sum())
        plates_red_m = int(((minute_events.side == 'RED') & (minute_events.action == 'PLATE')).sum())
        cum['plates_BLUE'] += plates_blue_m
        cum['plates_RED'] += plates_red_m

        # gold & cs per lane (ha van)
        # Ezek per lane szerinti cumulative team összegei (példa)
        gold_blue_total = None
        gold_red_total = None
        cs_blue_total = None
        cs_red_total = None
        if lane_gold_by_min:
            # feltételezve a labelsban mindkét team lane-jei vannak külön (pl. 'TOP','JGL',... kétszer — a te példa struktúrád duplicálja a role-kat a két csapatra)
            # A te struktúrádban nincs explicit side per dataset; ha a labels sorrendből tudod hogy első 6 a BLUE, második 6 RED -> ezt kell igazítani a valós adathoz
            # Itt egyszerűsítve összegzem az összes szereplő goldját (ha két csapat ad), különböző logika lehet szükséges a valós inputhoz
            try:
                gold_lists = list(lane_gold_by_min.values())
                # gold_lists: lista listák; összeadjuk index szerint
                gold_at_m = sum((lst[m] if m < len(lst) else 0) for lst in gold_lists)
                gold_blue_total = gold_at_m  # csak példa
            except Exception:
                gold_blue_total = None

        # create feature row
        row = {
            'game_id': game_id,
            'minute': m,
            'time_seconds': m*60,
            # cum stats
            'kills_BLUE_cum': cum['kills_BLUE'],
            'kills_RED_cum': cum['kills_RED'],
            'kills_diff_cum': cum['kills_BLUE'] - cum['kills_RED'],
            'towers_BLUE_cum': cum['towers_BLUE'],
            'towers_RED_cum': cum['towers_RED'],
            'towers_diff_cum': cum['towers_BLUE'] - cum['towers_RED'],
            'drakes_BLUE_cum': cum['drakes_BLUE'],
            'drakes_RED_cum': cum['drakes_RED'],
            'drakes_diff_cum': cum['drakes_BLUE'] - cum['drakes_RED'],
            'barons_BLUE_cum': cum['barons_BLUE'],
            'barons_RED_cum': cum['barons_RED'],
            'plates_BLUE_cum': cum['plates_BLUE'],
            'plates_RED_cum': cum['plates_RED'],
            # last objective time
            'min_since_last_drake': (m - last_obj_time['drake']) if last_obj_time['drake'] is not None else np.nan,
            'min_since_last_baron': (m - last_obj_time['baron']) if last_obj_time['baron'] is not None else np.nan,
            # if cs/gold available, include example fields
            'gold_total_est': gold_blue_total,
            # target (home side win?) -- itt egyszerűsítve: ha game_result_side == 'BLUE' akkor blue nyert
            'winner_side': game_result_side
        }
        rows.append(row)

    df_snap = pd.DataFrame(rows)
    # target boolean: home_win (például home=BLUE)
    # a te pipeline-odban legyen explicit: home_side per match
    df_snap['home_win'] = (df_snap['winner_side'] == 'BLUE')  # vagy paraméterezd
    # fontos: ne szivárogtass végkimenetet a featurebe (pl. ne add hozzá a végső célállapotot!)
    return df_snap

# Használat példa:
snapshots = build_snapshots_for_game(45323, events_raw, cs_gold, game_result_side='BLUE')
snapshots.head(10)
